In [1]:
%load_ext autoreload
%autoreload 2
import chromdyn
print("chromdyn path:", chromdyn.__file__)

chromdyn path: C:\Users\Leren\Documents\GitHub\chromdyn_new_stable\chromdyn\__init__.py


In [2]:
import os
import sys
#sys.path.append('../chromdyn')
from chromdyn.topology import TopologyGenerator
from chromdyn.chromatin_dynamics import ChromatinDynamics
from chromdyn.traj_utils import save_pdb

from chromdyn.hic_utils import HiCManager
hicman = HiCManager()

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [3]:
generator = TopologyGenerator()
n_beads_per_chain = 1
chain_num = 100

N = n_beads_per_chain * chain_num # total number of beads
# generate chain list
chain_list = [n_beads_per_chain for i in range(chain_num)]
type_list = []
block_length = 25
for i in range(N):
    if (i // block_length) % 2 == 0:
        type_list.append("A")
    else:
        type_list.append("B")

generator.gen_top(chain_list, type_list)

mass = 0.001
friction = 1.0
friction_phys = friction / mass

In [4]:
# Choose available platform
sim = ChromatinDynamics(generator.topology, name = 'testPBC', platform_name = "CUDA", output_dir = "output", console_stream = True, mass = mass)

#Add forces
kb = 80.0
lb = 1.0
sim.force_field_manager.add_harmonic_bonds(r0 = lb, k = kb,)# this function will automatically add bonds according to the topology file

kbound = 30.0
# note the formula of self avoidance here is different from OpenMiChroM
sim.force_field_manager.add_self_avoidance(Ecut = 5.0, k = 20.0, r = 1.0) #r = sigma = 2*radius

2026-05-01 20:57:54,306 | INFO | chromatin_dynamics | ************************************************************
2026-05-01 20:57:54,307 | INFO | chromatin_dynamics |         chromdyn v0.2.0.post39 : Chromatin Dynamics         
2026-05-01 20:57:54,308 | INFO | chromatin_dynamics | ************************************************************
2026-05-01 20:57:54,310 | INFO | chromatin_dynamics | System initialized with 100 particles. Output directory: output
2026-05-01 20:57:54,311 | INFO | platforms | Platform 'CUDA' is available and selected.
2026-05-01 20:57:54,311 | INFO | chromatin_dynamics | force_field_manager initialized. Use this to add forces before running setup.
2026-05-01 20:57:54,314 | INFO | forcefield | Adding 0 harmonic bonds with r0=1.0, k=80.0, group=0
2026-05-01 20:57:54,315 | INFO | forcefield | HarmonicBonds force successfully added to system.
2026-05-01 20:57:54,316 | INFO | forcefield | --------------------------------------------------
2026-05-01 20:57:54,318 |

In [5]:
block_size_prod = 200 # 0.1 tau_sim
T = 0.2
sim.simulation_setup(
    init_struct='randomwalk',
    integrator='active-langevin',
    temperature=T,
    timestep=0.0005,
    friction=friction_phys, # actually collision rate for any langevin type integrator in openmm
    save_pos=True,
    save_energy=True,
    energy_report_interval=5_000,
    pos_report_interval=block_size_prod,    
    PBC = True,
    box_vectors = (
        (10.0, 0.0, 0.0),
        (0.0, 10.0, 0.0),
        (0.0, 0.0, 10.0)
    ),   
    force_reinitialize = True
)

print("Simulating under temperature:", T)

2026-05-01 20:57:54,385 | INFO | forcefield | Updated nonbonded method for SelfAvoidance to Periodic.
2026-05-01 20:57:54,387 | INFO | integrators | Creating integrator ...
2026-05-01 20:57:54,388 | INFO | integrators | Active LangevinIntegrator: temperature=0.2 | friction=1000.0 | timestep=0.0005
2026-05-01 20:57:54,388 | INFO | integrators | Initialized active parameters: F=0.0 and t_corr=1.0.
2026-05-01 20:57:54,389 | INFO | integrators | These parameters are per dof variables and can be set any time using .set_active_params(F_seq, tau_seq)


Updated nonbonded method for SelfAvoidance to Periodic.


2026-05-01 20:57:55,492 | INFO | chromatin_dynamics | Setting up simulation context...
2026-05-01 20:57:55,493 | INFO | chromatin_dynamics | Random walk created. Position shape: (100, 3)
2026-05-01 20:57:55,495 | INFO | chromatin_dynamics | Simulation context initialized.
2026-05-01 20:57:55,496 | INFO | chromatin_dynamics | ------------------------------------------------------------------------------------------------------------------------
2026-05-01 20:57:55,497 | INFO | chromatin_dynamics | Index  Force Class                    Force Name           Group    Particles    Bonds        Exclusions   P.E./Particle       
2026-05-01 20:57:55,497 | INFO | chromatin_dynamics | ------------------------------------------------------------------------------------------------------------------------
2026-05-01 20:57:56,042 | INFO | chromatin_dynamics | 0      HarmonicBondForce              HarmonicBonds        0        N/A          0            N/A          0.000               
2026-05-01 20

Simulating under temperature: 0.2


In [6]:
F = 1.0
tau = 100.0

type_array = np.array(type_list)
F_seq = np.zeros(N)
F_seq = np.array((type_array == 'A').astype(int))*  F
print(F_seq)
tau_seq=[tau]*sim.num_particles

sim.set_activity(F_seq=F_seq, tau_seq=tau_seq)

2026-05-01 20:57:56,624 | INFO | chromatin_dynamics | Added active force and correlation times for 100 particles.


[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


In [7]:
simulation = sim.simulation
system = simulation.system
context = simulation.context
print(f"System uses PBC? {system.usesPeriodicBoundaryConditions()}")


System uses PBC? True


In [8]:
# step 3: relaxasion simulation
relaxation_steps = 100 * block_size_prod # 5000 tau_sim
sim.run(relaxation_steps, report=False)

'''reporter = sim.reporters['stability']
print(reporter.kinetic_threshold) # this should show 5.0
reporter.kinetic_threshold = 80.0
print(reporter.kinetic_threshold) # this should show 80.0'''
# we don't want velocities be reinitialized when sampling start
sim.reporters['stability'].force_reinitialize = False

2026-05-01 20:57:58,361 | INFO | chromatin_dynamics | ------------------------------------------------------------
2026-05-01 20:57:58,362 | INFO | chromatin_dynamics | Running simulation for 20000 steps...
2026-05-01 20:57:58,362 | INFO | chromatin_dynamics | Paused reporter: position
2026-05-01 20:57:58,363 | INFO | chromatin_dynamics | Paused reporter: energy
2026-05-01 20:58:00,438 | INFO | chromatin_dynamics | Completed 20000 steps in 2.07s (9641 steps/s)
2026-05-01 20:58:00,438 | INFO | chromatin_dynamics | ------------------------------------------------------------
2026-05-01 20:58:00,438 | INFO | chromatin_dynamics | Resumed reporter: position
2026-05-01 20:58:00,438 | INFO | chromatin_dynamics | Resumed reporter: energy


In [9]:
simulation = sim.simulation
system = simulation.system
context = simulation.context
print(f"System uses PBC? {system.usesPeriodicBoundaryConditions()}")

System uses PBC? True


In [10]:

n_blocks_prod = 1000
running_steps = n_blocks_prod * block_size_prod
v_history = []

for _ in range(n_blocks_prod):
    sim.run(block_size_prod, report=True)
    state = sim.simulation.context.getState(getVelocities=True)
    current_velocities = state.getVelocities(asNumpy=True)
    v_history.append(current_velocities)

# save velocity history
v_history = np.array(v_history)
np.save(os.path.join('output', f"velocity_history.npy"), v_history)
del v_history

# verify potential forces are added
sim.print_force_info()


2026-05-01 20:58:02,116 | INFO | chromatin_dynamics | ------------------------------------------------------------
2026-05-01 20:58:02,118 | INFO | chromatin_dynamics | Running simulation for 200 steps...
2026-05-01 20:58:02,146 | INFO | chromatin_dynamics | Completed 200 steps in 0.03s (7211 steps/s)
2026-05-01 20:58:02,149 | INFO | chromatin_dynamics | ------------------------------------------------------------
2026-05-01 20:58:02,150 | INFO | chromatin_dynamics | ------------------------------------------------------------
2026-05-01 20:58:02,152 | INFO | chromatin_dynamics | Running simulation for 200 steps...
2026-05-01 20:58:02,174 | INFO | chromatin_dynamics | Completed 200 steps in 0.02s (9090 steps/s)
2026-05-01 20:58:02,174 | INFO | chromatin_dynamics | ------------------------------------------------------------
2026-05-01 20:58:02,176 | INFO | chromatin_dynamics | ------------------------------------------------------------
2026-05-01 20:58:02,176 | INFO | chromatin_dynami

In [11]:
traj_file=sim.reporters.get('position').filename
#traj_file='output/testPBC_positions.cndb'
velocity_file = os.path.join('output', f"velocity_history.npy")


In [ ]:
# tools.py rely on OpenMiChroM.CndbTools, one may not choose to use it
r'''from tools import ndb2cndb, load, xyz
from OpenMiChroM.CndbTools import cndbTools
cndb_tool = cndbTools()
traj = load(cndb_tool, filename = traj_file)'''

In [12]:
from chromdyn.traj_utils import Trajectory
# in this new class, we can load the file and extract the trajectory with .load(), .xyz(), 
# or one can import load_trajectory(),get_xyz() function, they're the same, with self a Trajectory object.
traj = Trajectory(traj_file)
xyz = traj.xyz(frames=[0, None, 1], bead_selection=None)
xyz_wrapped = traj.xyz_wrapped()

Loaded output\testPBC_positions.cndb: 1000 frames, 100 beads.
Topology loaded: 100 atoms, 0 bonds
Box vectors loaded. Shape: (1000, 3, 3)


In [ ]:
# calculate Rg by types
#from chromdyn.traj_utils import Analyzer
rg_result = Trajectory.compute_rg_type(traj)
rg_result.keys()

In [ ]:
for key in rg_result.keys():
    print(key, np.mean(rg_result[key]), np.std(rg_result[key]))

rg_result['A'].shape

In [ ]:
rg_result = Trajectory.compute_rg_type(traj, get_components=True)
print(rg_result['general'].keys())

In [ ]:
l_p= Trajectory.compute_persistent_length_type(traj, custom_types=None)

In [ ]:
l_p

In [ ]:
from chromdyn.traj_utils import Analyzer
rdf_center = Analyzer.collect_distances_from_center_unified(
    coords_trajectory=xyz,
    bead_types=traj.chrom_seq,
    sampling_rate=1,
    batch_size=1000,
    device='gpu',
)

rdf_com = Analyzer.collect_distances_from_com_unified(
    coords_trajectory=xyz,
    bead_types=traj.chrom_seq,
    sampling_rate=1,
    batch_size=1000,
    device='gpu',
)

In [ ]:
rdf_center['A']

In [ ]:
plt.figure(figsize=(10, 6), dpi=100)
for key in rdf_center.keys():
    # Calculate the probability distribution function (PDF) using histogram
    data = rdf_center[key]
    
    # We use a histogram with density=True to get the probability density P(r)
    # The number of bins can be adjusted for smoothness
    counts, bin_edges = np.histogram(data, bins=100, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    # Plotting the distribution as a smooth line
    plt.plot(bin_centers, counts, label=f'Bead Type {key}', linewidth=2)
    plt.fill_between(bin_centers, counts, alpha=0.15)
# Formatting the plot
plt.xlabel('Distance from Center ($r$)', fontsize=12)
plt.ylabel('Probability Density $P(r)$', fontsize=12)
plt.title('Radial Probability Distribution Function', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Bead Types')
plt.tight_layout()
plt.show()

In [ ]:
# check if the box vectors are correctly saved, should be (Nframes, 3, 3)
np.shape(traj.box_vectors)

In [ ]:
traj.box_vectors

In [ ]:
# calculate PBC-HiC(image contact included)
hic = hicman.gen_pbc_hic_from_cndb(traj_file = traj_file, mu = 3.22, rc = 1.78,p = None, platform = 'CUDA', batch_size = 100)

In [ ]:
print("Analysing trajectory file:", traj.cndb)

# getatom  = getresidue for our case, there're all number of beads
# traj.topology.getNumAtoms()
# traj.topology.getNumResidues()
print("Topology: ",traj.topology)
print("Number of chains: ", traj.topology.getNumChains())
print("Number of frames: ", traj.n_frames)
print("Number of beads: ", traj.n_beads)
print("Chain info: ",traj.chain_info)
print("Unique chromatin Types: ",traj.unique_chrom_seq)
print("Chromatin sequence: ",traj.chrom_seq)
print("Dictionary of chromatin sequence: ",traj.dict_chrom_seq)

In [ ]:
traj.chrom_seq

In [ ]:
print(traj.chrom_seq)
bead_types_raw = traj.chrom_seq
if bead_types_raw and isinstance(bead_types_raw[0], bytes):
    bead_types = np.array([b.decode('utf-8') for b in bead_types_raw])
else:
    bead_types = np.asarray(bead_types_raw)

print(bead_types)

In [ ]:
rg_result = Trajectory.compute_rg_type(traj, get_components=True)
# analysis of general rg
print("box basis: ", traj.box_vectors[0]) # box basis was saved to each frame
print("mean rg: ",np.mean(rg_result["general"]["total"]))
print("std rg: ",np.std(rg_result["general"]["total"]))
print("mean Rg by 3 directions: ",np.mean(rg_result["general"]["components"], axis=0))
print("std Rg by 3 directions: ",np.std(rg_result["general"]["components"], axis=0))


In [ ]:
import inspect

# print the source code of traj.xyz
print(inspect.getsource(traj.xyz))
print(traj.xyz.__code__.co_filename)

In [ ]:
from chromdyn.visualization import visualize, visualize_animation
# these 2 visualization function are for the new format of cndb files with pbc mode on
pbc_box_side_length = 10
visualize(
     traj = traj,
     select_frame=532,
     axis_limits=(-10, pbc_box_side_length + 10, -10, pbc_box_side_length + 10, -10, pbc_box_side_length + 10), # Optional
     colors=None, # Optional
     isring=False,
     r=0.5,
     PBC=True,
     color_mode='type'
)



In [ ]:
from chromdyn.visualization import visualize_pbc_images
visualize_pbc_images(
    traj = traj,
    select_frame = 532,
    n_layers = 1, # 3*3*3 grid
    image_alpha = 0.15,
    image_style = 'scatter',
    r = 0.5,
    color_mode = 'type'
)

In [ ]:
plt.rcParams['animation.html'] = 'jshtml'
pbc_box_side_length = 10
visualize_animation(
     traj = traj,
     #box_a=pbc_box_side_length,
     start_frame = 0,
     end_frame = 300,
     axis_limits=(-10, pbc_box_side_length + 10, -10, pbc_box_side_length + 10, -10, pbc_box_side_length + 10), # Optional
     colors=None, # Optional
     isring=False,
     r=0.5,
     #output_name = 'output/animation_pbc.mp4',
     PBC=True
 )

In [ ]:
# velocity correlation checking
velocity = np.load(velocity_file)   
#velocity

In [ ]:
# There're two functions that calculate time and spatial coorelation of velocity inside Cndbtools.py, using GPU acceleration
from chromdyn.traj_utils import Analyzer
bead_types = np.array(traj.chrom_seq)
v_correlation = Analyzer.calculate_vacf(
    velocities = velocity,
    bead_types = bead_types,
    sampling_step=1,
    platform='CPU'
)

v_spatial_correlation = Analyzer.calculate_spatial_vel_corr(
    coords = xyz,
    velocities = velocity,
    bead_types = bead_types,
    sampling_step=1,
    dist_range=10.0,
    num_bins=50,
    platform='CPU'
)

In [ ]:
# Each output is a dictionary with the correlation function for general and each bead type
# verify format if v_correlation
print(v_correlation.keys())
# plot the correlation function (time series)
v_plot = v_correlation['general']
import matplotlib.pyplot as plt

plt.figure()
plt.plot(v_plot)
plt.xlabel(r'Time lag(0.1 $\tau$)')
plt.ylabel('Velocity Autocorrelation')
plt.xlim(0,100)
plt.title('General Velocity Autocorrelation Function')
plt.show()



In [ ]:
# The output is a dictionary with the correlation function for general and each bead type
# verify format if v_correlation
print(v_spatial_correlation.keys())
# plot spatial correlation, x axis  = v_spatial_correlation['bin_centers'], y axis = ['general']
plt.figure(figsize=(10, 6))
plt.plot(v_spatial_correlation['bin_centers'], v_spatial_correlation['general'], label='General Correlation')
plt.xlabel('Distance')
plt.ylabel('Correlation')
plt.title('Spatial Correlation Function')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from chromdyn.traj_utils import Analyzer
msd = Analyzer.compute_msd(
    positions = xyz,
    batch_size = 1000,
    platform = 'auto'
)

In [ ]:
# verify size of msd: should be (num_of_frame, num_of_beads)
print(msd.shape)
avg_msd = np.mean(msd, axis= 1) # average over all beads

plt.figure()
plt.plot(avg_msd)
plt.xlabel(r'Time lag(0.1 $\tau$)')
plt.ylabel('MSD')
plt.xlim(0,1000)
plt.title('MSD')
plt.show()